# 🎓 캡스톤 — 손실함수를 갈아 끼우다 (5부 대단원)

"수포자를 위한" 시리즈 5부(정보이론)의 마지막 실습입니다. 두 가지를 해냅니다:

- **실험 A · 손실 교체**: 1부 캡스톤의 동심원 분류 신경망을 그대로 가져와,
  MSE 를 **크로스 엔트로피(BCE)** 로 교체합니다. 1부에서 "받아 적은 선물"이었던
  σ′ 이 **소거되어 기울기가 (ŷ − y) 로 단순해지는 마법**을 수치미분으로 검산하고
  (ŷ 는 **와이-햇**, 곧 모델의 예측값. 햇 ^ 은 "추정한 값"이라는 표기 관례예요),
  두 손실의 학습 곡선을 대결시킵니다.
- **실험 B · 미니 언어모델**: 문자 단위 bigram 모델을 numpy 로 만들어
  실제 텍스트의 **퍼플렉시티**를 재고, **temperature 샘플링**으로 문장을 생성합니다.
  4부에서 "그렇다더라"로 배운 것들을 전부 손수 재현.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch10')

## 실험 A · 손실 교체: 1부의 신경망을 소환한다

### A-1. 데이터와 네트워크 (1부 캡스톤 그대로)

직선으로 못 나누는 두 무리: 안쪽 뭉치(y=0)와 바깥 고리(y=1),
그리고 2 → 8 → 1 미니 신경망(가중치 33개).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

rng = np.random.default_rng(42)
n = 100                                   # 무리당 점 개수

# 안쪽 뭉치 / 바깥 고리 — 1부 캡스톤과 같은 데이터
r = rng.uniform(0, 1.1, n); theta = rng.uniform(0, 2 * np.pi, n)
inner = np.c_[r * np.cos(theta), r * np.sin(theta)]
r = rng.uniform(1.9, 2.8, n); theta = rng.uniform(0, 2 * np.pi, n)
outer = np.c_[r * np.cos(theta), r * np.sin(theta)]

X = np.vstack([inner, outer])
y = np.vstack([np.zeros((n, 1)), np.ones((n, 1))])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X, W1, b1, W2, b2):
    """순전파 — 1부 캡스톤 그대로: 행렬곱 → tanh → 행렬곱 → 시그모이드."""
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    Z2 = A1 @ W2 + b2
    y_hat = sigmoid(Z2)
    return Z1, A1, Z2, y_hat

def init_params(seed=7):
    """두 손실을 같은 출발선에서 대결시키기 위한 공통 초기화."""
    r = np.random.default_rng(seed)
    return (r.normal(0, 0.8, (2, 8)), np.zeros((1, 8)),
            r.normal(0, 0.8, (8, 1)), np.zeros((1, 1)))

W1, b1, W2, b2 = init_params()
_, _, _, y_hat = forward(X, W1, b1, W2, b2)
print(f'학습 전 정확도: {np.mean((y_hat > 0.5) == y):.1%}, 동전 던지기 수준 (1부의 그 출발점)')

### A-2. 두 손실, 두 역전파: σ′ 소거의 마법

출력층의 책임(델타)을 비교하면 이 캡스톤의 심장이 보입니다:

| 손실 | 출력층 델타 | 특징 |
|------|------------|------|
| **MSE** (1부) | $\frac{2}{N}(\hat{y}-y)\cdot\sigma'(z)$ | σ′ = σ(1−σ) 가 곱해져 확신 구간에서 **기울기 사망** |
| **BCE** (5부) | $\frac{1}{N}(\hat{y}-y)$ | σ′ 이 **정확히 소거**되어 틀린 만큼 그대로 배움 |

1부에서 σ′ = σ(1−σ) 를 "받아 적은 선물"이라 불렀죠. 5부의 −log 손실을 만나면
그 선물이 분모와 약분되어 사라집니다. 믿지 말고 **수치미분으로 검산**합시다.

In [ ]:
def mse_loss(y_hat, y):
    return np.mean((y_hat - y) ** 2)

def bce_loss(y_hat, y):
    """이진 크로스 엔트로피 — 원핫 CE 의 두 클래스 버전. -log q(정답) 의 평균 (nat)."""
    eps = 1e-12                            # log 0 방지 (4장: q=0 이면 비용 무한대!)
    return float(np.mean(-(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))))

def backward(X, y, W2, A1, y_hat, kind):
    """역전파 — 1부 캡스톤과 같은 구조, 출력층 델타만 손실에 따라 다르다."""
    N = len(X)
    if kind == 'mse':
        delta2 = 2 * (y_hat - y) / N * y_hat * (1 - y_hat)   # σ′ 이 곱해진다
    else:  # bce
        delta2 = (y_hat - y) / N                              # σ′ 이 소거됐다!
    dW2 = A1.T @ delta2
    db2 = delta2.sum(axis=0, keepdims=True)
    delta1 = delta2 @ W2.T * (1 - A1 ** 2)
    dW1 = X.T @ delta1
    db1 = delta1.sum(axis=0, keepdims=True)
    return dW1, db1, dW2, db2

# ── 수치미분 검산 (1부 ch12 방식 그대로) — BCE 역전파가 정말 맞는지 ──
W1, b1, W2, b2 = init_params()
_, A1, _, y_hat = forward(X, W1, b1, W2, b2)
dW1, db1, dW2, db2 = backward(X, y, W2, A1, y_hat, kind='bce')

def bce_at(W1, b1, W2, b2):
    return bce_loss(forward(X, W1, b1, W2, b2)[3], y)

h = 1e-6
print('가중치      역전파(δ=ŷ−y)   수치미분        차이')
for name, P, dP, idx in [('W1[0,0]', W1, dW1, (0, 0)), ('W2[3,0]', W2, dW2, (3, 0))]:
    P[idx] += h;      L_plus = bce_at(W1, b1, W2, b2)
    P[idx] -= 2 * h;  L_minus = bce_at(W1, b1, W2, b2)
    P[idx] += h
    numeric = (L_plus - L_minus) / (2 * h)
    print(f'{name}  {dP[idx]:13.8f}  {numeric:13.8f}  {abs(dP[idx] - numeric):.1e}')
    assert abs(dP[idx] - numeric) < 1e-5

print('\n✅ 검산 통과! σ′ 없이 (ŷ−y) 만으로 정확한 기울기입니다. 소거는 진짜였어요.')

### A-3. 학습 대결: 같은 출발선, 다른 손실

같은 초기 가중치에서 MSE 팀과 BCE 팀을 각각 2,000걸음 경사하강시킵니다.

In [ ]:
def train(kind, eta=0.8, epochs=2000):
    W1, b1, W2, b2 = init_params()          # 공통 출발선
    accs = []
    for _ in range(epochs):
        _, A1, _, y_hat = forward(X, W1, b1, W2, b2)
        dW1, db1, dW2, db2 = backward(X, y, W2, A1, y_hat, kind)
        W1 -= eta * dW1;  b1 -= eta * db1
        W2 -= eta * dW2;  b2 -= eta * db2
        accs.append(float(np.mean((y_hat > 0.5) == y)))
    return np.array(accs)

acc_mse = train('mse')
acc_bce = train('bce')

def first_reach(accs, level=0.99):
    hit = np.argmax(accs >= level)
    return int(hit) if accs[hit] >= level else None

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(acc_mse * 100, color='#f59e0b', lw=2, label='MSE (1부의 우회)')
ax.plot(acc_bce * 100, color='#6366f1', lw=2, label='크로스 엔트로피 (5부의 정공법)')
ax.set_xlabel('에포크 (전체 데이터 한 바퀴)')
ax.set_ylabel('정확도 (%)')
ax.set_title('같은 신경망, 같은 출발선: 손실만 갈아 끼운 대결')
ax.grid(True, color='#eef2f7')
ax.legend()
plt.show()

for name, accs in [('MSE', acc_mse), ('BCE', acc_bce)]:
    r = first_reach(accs)
    print(f'{name}: 최종 정확도 {accs[-1]:.1%}, 정확도 99% 도달 = {r if r is not None else "미도달"}걸음')
assert acc_bce[-1] >= 0.99, 'BCE 팀이 과녁을 못 맞혔다?'

### A-4. 왜 차이가 나나: 죽은 기울기의 현장

결정적 차이는 **확신에 찬 오답** 구간에 있습니다. 정답 y=1 인데 모델이 ŷ→0 으로
확신하며 틀릴 때, 출력층 기울기의 크기를 비교해 보세요.

In [ ]:
y_hat_range = np.linspace(0.001, 0.999, 400)
y_true = 1.0

grad_mse = np.abs(2 * (y_hat_range - y_true) * y_hat_range * (1 - y_hat_range))  # σ′ 포함
grad_bce = np.abs(y_hat_range - y_true)                                          # σ′ 소거

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y_hat_range, grad_bce, color='#6366f1', lw=2.5, label="BCE 기울기 |예측 - 정답|")
ax.plot(y_hat_range, grad_mse, color='#f59e0b', lw=2.5, label="MSE 기울기 (시그마 도함수 곱해짐)")
ax.annotate('확신에 찬 오답 (예측 0, 정답 1)\nBCE: 기울기 1 - 크게 배움\nMSE: 기울기 0 - 학습 마비',
            (0.02, 0.95), fontsize=10, va='top')
ax.set_xlabel('모델의 예측 (정답 y = 1)')
ax.set_ylabel('출력층 기울기의 크기')
ax.set_title('시그마 도함수가 곱해지면 양 끝에서 기울기가 죽는다 - 소거의 가치')
ax.grid(True, color='#eef2f7')
ax.legend(loc='center right')
plt.show()

print(f'예측 0.001 에서 MSE 기울기: {grad_mse[0]:.6f} (마비) vs BCE 기울기: {grad_bce[0]:.3f} (건강)')

## 실험 B · 미니 언어모델: bigram 으로 GPT 흉내내기

### B-1. 모델 만들기: "앞 글자를 보고 다음 글자를 맞혀라"

9장에서 쟀던 그 구조를 **모델**로 만듭니다: 텍스트에서 글자쌍을 세어
조건부 분포 P(다음 글자 | 앞 글자) 표를 얻으면 끝. 이것이 bigram 언어모델입니다.
훈련 = 세기(6장의 최대우도가 이 경우 그냥 빈도 세기와 같아요!).

In [ ]:
corpus = ('information is the resolution of uncertainty. the amount of surprise of an '
          'event depends on how rare it is. rare events carry more information than '
          'common ones. a good question quickly cuts the candidates in half. entropy '
          'measures the average surprise of a distribution. a fair coin has one bit of '
          'entropy and a loaded coin has less. cross entropy is the cost of using the '
          'wrong map of the world. the excess cost is the divergence. learning is the '
          'art of making the model match the data. perplexity turns the loss back into '
          'the number of choices the model is juggling. temperature reshapes the '
          'distribution before sampling. context reduces uncertainty and attention '
          'finds the context that matters most.')

chars = sorted(set(corpus))
V = len(chars)
idx = {c: i for i, c in enumerate(chars)}
print(f'말뭉치 {len(corpus)}글자, 어휘 {V}종류: {"".join(chars)!r}')

# 훈련 = 글자쌍 세기 (+ 아주 작은 스무딩 0.01 — 못 본 쌍에도 미세한 확률을 남긴다.
#   4장에서 배웠죠: 일어날 수 있는 일에 q=0 을 주면 비용이 무한대!)
counts = np.full((V, V), 0.01)
for prev, nxt in zip(corpus[:-1], corpus[1:]):
    counts[idx[prev], idx[nxt]] += 1
bigram = counts / counts.sum(axis=1, keepdims=True)   # 행 = 앞 글자, 열 = 다음 글자 확률

# 비교용 unigram 모델(문맥 무시)과 uniform 모델(아무것도 모름)
uni_counts = np.full(V, 0.01)
for c in corpus:
    uni_counts[idx[c]] += 1
unigram = uni_counts / uni_counts.sum()

row = bigram[idx['q']]
print(f"모델이 배운 것: 'q' 다음은 '{chars[int(row.argmax())]}' {row.max()*100:.0f}%  (9장에서 실측한 그 규칙!)")

### B-2. 성적표: 세 모델의 퍼플렉시티 (7장 실전)

같은 텍스트를 세 모델로 채점합니다: 글자마다 −log₂ q(정답) → 평균 → 2^.
문맥을 쓸수록(9장) 성적이 좋아지는 걸 확인하세요.

In [ ]:
def ppl_uniform():
    return float(V)                       # 균등 V갈래의 PPL = V (7장)

def ppl_unigram():
    losses = [-np.log2(unigram[idx[c]]) for c in corpus]
    return float(2 ** np.mean(losses))

def ppl_bigram():
    losses = [-np.log2(bigram[idx[p], idx[nx]]) for p, nx in zip(corpus[:-1], corpus[1:])]
    return float(2 ** np.mean(losses))

ppl_u, ppl_1, ppl_2 = ppl_uniform(), ppl_unigram(), ppl_bigram()

print('모델                     | PPL (평균 몇 갈래 고민)')
print('-' * 48)
print(f'🤷 uniform (아무것도 모름) | {ppl_u:6.2f} 갈래')
print(f'📊 unigram (빈도만 앎)     | {ppl_1:6.2f} 갈래')
print(f'🔗 bigram  (앞 글자 봄)    | {ppl_2:6.2f} 갈래')
print('\n빈도를 알면 성적이 오르고(2~3장), 문맥을 보면 또 오른다(9장).')
print('GPT 는 이 사다리를 "앞 글자 하나"에서 "앞 토큰 수십만 개"로 늘린 것입니다.')
assert ppl_2 < ppl_1 < ppl_u

### B-3. 생성: temperature 다이얼을 돌리며 (8장 실전)

이제 이 미니 모델로 **문장을 생성**합니다. 앞 글자의 조건부 분포에 temperature 를
적용해 다음 글자를 뽑기를 반복합니다. GPT 의 생성 루프 그 자체입니다(모델만 초소형).

In [ ]:
def generate(T=1.0, length=150, start='t', seed=123):
    r = np.random.default_rng(seed)
    out = start
    for _ in range(length):
        p = bigram[idx[out[-1]]] ** (1 / T)      # temperature — 로짓/T 와 같은 효과 (8장)
        p = p / p.sum()
        out += chars[int(r.choice(V, p=p))]
    return out

for T in [0.5, 1.0, 2.0]:
    print(f'--- T = {T} ---')
    print(generate(T))
    print()

print('🧊 낮은 T: 흔한 패턴(the, tion)만 반복 · 🔥 높은 T: 알파벳 잔치.')
print('bigram 이라 어차피 말은 안 되지만, 다이얼의 효과는 GPT 와 정확히 같은 원리입니다.')

## 직접 풀어보기 ✏️: 마지막 문제

세 모델(uniform / unigram / bigram) 중 성적(퍼플렉시티)이 **가장 좋은** 모델은
무엇이고, 왜일까요?

In [ ]:
# TODO: answer 를 'uniform' / 'unigram' / 'bigram' 중 하나로 바꾸고 실행하세요.
# 힌트: 성적이 좋다 = PPL 이 낮다. 문맥(앞 글자)은 불확실성을 줄인다(9장).

answer = 'bigram'
ppls = {'uniform': ppl_u, 'unigram': ppl_1, 'bigram': ppl_2}
print(f'내 답: {answer} (PPL {ppls[answer]:.2f})')
assert ppls[answer] == min(ppls.values()), '다시 생각해 보세요 — 힌트를 많이 쓰는 모델일수록?'
print('✅ 정답! 문맥을 쓰는 모델이 가장 덜 헷갈립니다. I(다음 글자; 앞 글자) > 0 이니까요.')

---
## 🎓 5부 대단원: 이제 답할 수 있는 네 가지

1. **AI 의 손실함수는 왜 크로스 엔트로피인가?** 학습 = 데이터 분포 흉내내기(KL 최소화)이고,
   이것이 곧 CE 최소화이며(6장), 시그모이드·소프트맥스와 만나면 기울기까지 (ŷ−y) 로
   단순해진다(오늘 검산!). 1부에서 MSE 로 우회했던 복선이 완전히 풀렸습니다.
2. **퍼플렉시티는 무엇인가?** 2^손실 = "평균 몇 갈래에서 고민하나"(7장). 오늘 세 모델로 직접 쟀다.
3. **temperature 는 무엇인가?** softmax(z/T), 분포의 엔트로피 다이얼(8장). 오늘 생성으로 돌려 봤다.
4. **RLHF 의 KL 페널티는 무엇인가?** 원본 모델 분포에서 멀어지지 않게 잡는 목줄, 초과 비트의 벌점(5장).

**시리즈의 한 줄**: AI 는 불확실성을 줄이는 기계이고(워밍업), 배운다는 것은 세상 분포를
흉내내는 것이며(6장), 그 모든 눈금은 놀람 log₂(1/p) 하나에서 나왔다(1장).

1부(선형대수) → 2부(미분) → 3부(확률) → 4부(트랜스포머) → 5부(정보이론), 여기까지 완주를 축하합니다! 🎉